# Data Preparation

1. Bug fixes in the existing filter definitions
2. Column standardisation (`text` + `label` for every dataset piece)
3. Benign row collection per SLM
4. Merge injection + benign, deduplicate, class-balance at 3:1
5. Gemma 3 chat-template formatting with explicit `<bos>` and model answer
6. Prompt-template variation to reduce template overfitting
7. Stratified 80/10/10 train/val/test split while retaining labels
8. Push all three final datasets to HuggingFace


In [54]:
import os
# Disable the harmless symlink warning on Windows (caching still works fine)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
from datasets import DatasetDict, Dataset, load_dataset
from huggingface_hub import login
import pandas as pd

In [75]:
HF_TOKEN = None

# Kaggle secret support
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("Loaded HF_TOKEN from Kaggle secrets.")
except Exception:
    pass

# Colab secret support
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("Loaded HF_TOKEN from Colab secrets.")
    except Exception:
        pass

if HF_TOKEN is None:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path='./../.env')
    HF_TOKEN = os.environ.get("HF_TOKEN")
    login(token=HF_TOKEN)
else:
    login()  # interactive prompt

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [56]:
INJECTION_COLUMN_NAME = 'injection_category'

print('Loading datasets...')
raw = [
    load_dataset('neuralchemy/Prompt-injection-dataset',        'full'),        # [0]
    load_dataset('neuralchemy/prompt-injection-Threat-Matrix',  'multiclass'),  # [1]
    load_dataset('watchdogsrox/Mirror-Prompt-Injection-Dataset', split='train'),# [2]
    load_dataset('Abdennebi/synthetic-prompt-injections',        split='train'),# [3]
    load_dataset('Abdennebi/shieldlm-prompt-injection',          split='train'),# [4]
    load_dataset('centrepourlasecuriteia/jailbreak-dataset',     split='train'),# [5]  ← was [6]
]
print('All datasets loaded.')

Loading datasets...
All datasets loaded.


In [57]:
# RENAME category columns to a common name
# neuralchemy/Prompt-injection-dataset  → 'category'
# neuralchemy/Threat-Matrix             → 'intent'
# Mirror                                → 'category'
# synthetic                             → 'category'
# shieldlm                              → 'label_category'
# centrepour                            → 'technique_type

def rename_split(ds_or_dict, old, new):
    if hasattr(ds_or_dict, 'rename_column'):
        return ds_or_dict.rename_column(old, new)
    return {k: v.rename_column(old, new) for k, v in ds_or_dict.items()}

raw[0] = rename_split(raw[0], 'category', INJECTION_COLUMN_NAME)
raw[1] = rename_split(raw[1], 'intent', INJECTION_COLUMN_NAME)
raw[2] = rename_split(raw[2], 'category', INJECTION_COLUMN_NAME)
raw[3] = rename_split(raw[3], 'category', INJECTION_COLUMN_NAME)
raw[4] = rename_split(raw[4], 'label_category', INJECTION_COLUMN_NAME)
raw[5] = rename_split(raw[5], 'technique_type', INJECTION_COLUMN_NAME)

print('Column rename done.')

Column rename done.


In [58]:
# Create 3 datasets for each SLM 

# Helper: get the 'train' split whether the object is a DatasetDict or a Dataset
def train(ds):
    return ds['train'] if hasattr(ds, '__getitem__') and 'train' in ds else ds

dataset_a_pieces = [
    train(raw[0]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'direct_injection', 'jailbreak', 'persona_replacement',
        'many_shot', 'crescendo',
    ]),

    train(raw[1]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'role_hijack', 'direct_injection',
    ]),
    
    train(raw[2]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'ignore', 'persona',
    ] and x['label'] == 1),
    
    train(raw[4]).filter(lambda x: x[INJECTION_COLUMN_NAME] == 'jailbreak'),
    
    raw[5].filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'cognitive_psychological', 'fsh', 'dap',
    ] and x['category'] != 'Benign'),
]

dataset_b_pieces = [
    train(raw[0]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'system_extraction', 'prompt_leaking', 'indirect_injection',
    ]),
    
    train(raw[1]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'system_extraction', 'tool_abuse', 'indirect_injection',
    ]),
    
    train(raw[2]).filter(lambda x: x[INJECTION_COLUMN_NAME] == 'extraction'
                         and x['label'] == 1),
    
    train(raw[4]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'direct_injection', 'indirect_injection',
    ]),
]

dataset_c_pieces = [
    train(raw[0]).filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'encoding_obfuscation', 'token_smuggling', 'context_overflow',
    ]),
    
    train(raw[1]).filter(lambda x: x[INJECTION_COLUMN_NAME] == 'obfuscation'),
    
    raw[3].filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'code_obfuscation', 'typoglycemia',
    ] and str(x['label']) == '1'),
    
    raw[5].filter(lambda x: x[INJECTION_COLUMN_NAME] in [
        'encoding_cyphering', 'structural_obfuscation',
        'ascii_art', 'tokenbreak', 'adversarial_suffixes',
    ] and x['category'] != 'Benign'),
]

print('Datasets divided.')

Datasets divided.


# Standardise columns to `text` + `label` (int)

Every dataset piece has different column names and label types.  
This function normalises all of them to exactly two columns:
- `text` — the raw prompt string
- `label` — integer, 1 = injection, 0 = benign

In [59]:
def standardise(ds, source_name):
    """
    Returns a pandas DataFrame with exactly two columns: text (str), label (int).
    Handles all column-name and label-type quirks across the 6 source datasets.
    """
    df = ds.to_pandas()

    # Resolve text column
    if 'jailbreak_prompt' in df.columns:
        # centrepourlasecuriteia: the TRANSFORMED prompt is the attack text
        df = df.rename(columns={'jailbreak_prompt': 'text'})
    elif 'prompt' in df.columns and 'text' not in df.columns:
        df = df.rename(columns={'prompt': 'text'})
    # else: 'text' already present (neuralchemy, mirror, synthetic, shieldlm)

    if 'binary_label' in df.columns:
        # Threat-Matrix multiclass config has BOTH 'label' (7-class intent int)
        # AND 'binary_label' (0/1). Drop the 7-class 'label' first so the
        # rename does not create two columns both called 'label', which makes
        # pandas return a 2D DataFrame for df['label'] → crashes value_counts().
        if 'label' in df.columns:
            df = df.drop(columns=['label'])
        df = df.rename(columns={'binary_label': 'label'})
    elif 'label' not in df.columns:
        # centrepour has no binary label — derive from category column
        if 'category' in df.columns:
            df['label'] = (df['category'] != 'Benign').astype(int)
        elif 'label_binary' in df.columns:
            df = df.rename(columns={'label_binary': 'label'})
        else:
            raise ValueError(f'Cannot resolve label for {source_name}')
    
    # Cast label to int (synthetic uses string '0'/'1')
    df['label'] = df['label'].astype(int)

    # Keep only the two required columns
    df = df[['text', 'label']].copy()

    # Drop rows with null/empty text
    df = df.dropna(subset=['text'])
    df = df[df['text'].str.strip() != '']

    print(f'  {source_name}: {len(df)} rows | label dist: {df["label"].value_counts().to_dict()}')
    return df

print('standardise() defined.')

standardise() defined.


In [60]:
# Apply standardise() to every injection piece
SOURCE_NAMES_A = [
    'neuralchemy-full[A]', 'threat-matrix[A]', 'mirror[A]',
    'shieldlm[A]', 'centrepour[A]',
]
SOURCE_NAMES_B = [
    'neuralchemy-full[B]', 'threat-matrix[B]', 'mirror[B]', 'shieldlm[B]',
]
SOURCE_NAMES_C = [
    'neuralchemy-full[C]', 'threat-matrix[C]', 'synthetic[C]', 'centrepour[C]',
]

print('=== SLM A: Instruction & Role Violation Detector ===')
a_inj_dfs = [standardise(ds, n) for ds, n in zip(dataset_a_pieces, SOURCE_NAMES_A)]

print('\n=== SLM B: Privilege Escalation Detector ===')
b_inj_dfs = [standardise(ds, n) for ds, n in zip(dataset_b_pieces, SOURCE_NAMES_B)]

print('\n=== SLM C: Obfuscation & Evasion Pattern Detector ===')
c_inj_dfs = [standardise(ds, n) for ds, n in zip(dataset_c_pieces, SOURCE_NAMES_C)]

=== SLM A: Instruction & Role Violation Detector ===
  neuralchemy-full[A]: 5890 rows | label dist: {1: 5890}
  threat-matrix[A]: 9136 rows | label dist: {1: 9136}
  mirror[A]: 3315 rows | label dist: {1: 3315}
  shieldlm[A]: 712 rows | label dist: {1: 712}
  centrepour[A]: 1860 rows | label dist: {1: 1860}

=== SLM B: Privilege Escalation Detector ===
  neuralchemy-full[B]: 72 rows | label dist: {1: 72}
  threat-matrix[B]: 7920 rows | label dist: {1: 7920}
  mirror[B]: 1260 rows | label dist: {1: 1260}
  shieldlm[B]: 12563 rows | label dist: {1: 12563}

=== SLM C: Obfuscation & Evasion Pattern Detector ===
  neuralchemy-full[C]: 115 rows | label dist: {1: 115}
  threat-matrix[C]: 2296 rows | label dist: {1: 2296}
  synthetic[C]: 33064 rows | label dist: {1: 33064}
  centrepour[C]: 3100 rows | label dist: {1: 3100}


# Collect benign rows per SLM

Benign rows are drawn from multiple sources and allocated per SLM as planned.  
The target is **3× the injection count** for each SLM (3:1 ratio).  
`tatsu-lab/alpaca` fills any remaining gap.

In [61]:
# Concatenate injection pieces per SLM

inj_a = pd.concat(a_inj_dfs, ignore_index=True)
inj_b = pd.concat(b_inj_dfs, ignore_index=True)
inj_c = pd.concat(c_inj_dfs, ignore_index=True)

# Keep only label=1 rows (all injection pieces should be 1, but guard anyway)
inj_a = inj_a[inj_a['label'] == 1]
inj_b = inj_b[inj_b['label'] == 1]
inj_c = inj_c[inj_c['label'] == 1]

# Deduplicate by exact text
inj_a = inj_a.drop_duplicates(subset='text').reset_index(drop=True)
inj_b = inj_b.drop_duplicates(subset='text').reset_index(drop=True)
inj_c = inj_c.drop_duplicates(subset='text').reset_index(drop=True)

print(f'Injections after dedup — A: {len(inj_a):,}  B: {len(inj_b):,}  C: {len(inj_c):,}')

# SLM C: cap at 20k with stratified sample across centrepour technique types
SLM_C_INJ_CAP = 20_000
if len(inj_c) > SLM_C_INJ_CAP:
    inj_c = inj_c.sample(n=SLM_C_INJ_CAP, random_state=42).reset_index(drop=True)
    print(f'SLM C injections capped to {SLM_C_INJ_CAP:,}')

print(f'Final injection counts — A: {len(inj_a):,}  B: {len(inj_b):,}  C: {len(inj_c):,}')

Injections after dedup — A: 16,870  B: 20,890  C: 38,552
SLM C injections capped to 20,000
Final injection counts — A: 16,870  B: 20,890  C: 20,000


In [62]:
# Collect benign rows from each source

# neuralchemy/Prompt-injection-dataset benign (label=0, all splits)
neur_benign = standardise(
    train(raw[0]).filter(lambda x: x['label'] == 0),
    'neuralchemy-benign'
)

# Threat-Matrix benign (binary_label=0)
tm_benign = standardise(
    train(raw[1]).filter(lambda x: x['binary_label'] == 0),
    'threat-matrix-benign'
)

# Mirror benign (label=0) — hard negatives with dangerous keywords in benign context
mirror_benign = standardise(
    train(raw[2]).filter(lambda x: x['label'] == 0),
    'mirror-benign'
)

# synthetic benign (label='0', category='uncategorized') — general instruction text
synth_benign = standardise(
    raw[3].filter(lambda x: str(x['label']) == '0'),
    'synthetic-benign'
)

# shieldlm benign (label_category='benign') — multilingual + tool-response text
shieldlm_benign = standardise(
    train(raw[4]).filter(lambda x: x[INJECTION_COLUMN_NAME] == 'benign'),
    'shieldlm-benign'
)

# centrepour benign (category='Benign') — JailbreakBench goals, hard negatives
centrepour_benign = standardise(
    raw[5].filter(lambda x: x['category'] == 'Benign'),
    'centrepour-benign'
)

# 7. tatsu-lab/alpaca — general instruction text, gap filler
alpaca_raw = load_dataset('tatsu-lab/alpaca', split='train')
alpaca_df = pd.DataFrame({'text': alpaca_raw['instruction'], 'label': 0})
alpaca_df = alpaca_df[alpaca_df['text'].str.strip() != ''].drop_duplicates(subset='text')
print(f'alpaca-benign: {len(alpaca_df):,} rows')

  neuralchemy-benign: 5208 rows | label dist: {0: 5208}
  threat-matrix-benign: 6504 rows | label dist: {0: 6504}
  mirror-benign: 4995 rows | label dist: {0: 4995}
  synthetic-benign: 126423 rows | label dist: {0: 126423}
  shieldlm-benign: 24638 rows | label dist: {0: 24638}
  centrepour-benign: 899 rows | label dist: {0: 899}
alpaca-benign: 52,002 rows


In [63]:
# Allocate benign rows per SLM (proportional allocation + alpaca gap fill)

def sample_up_to(df, n, random_state=42):
    """Sample min(n, len(df)) rows."""
    return df.sample(n=min(n, len(df)), random_state=random_state)

In [64]:
# SLM B: Privilege Escalation Detector benign allocation
target_b = len(inj_b) * 3
ben_b = pd.concat([
    sample_up_to(neur_benign,       3_000),
    sample_up_to(tm_benign,        10_000),
    sample_up_to(mirror_benign,     4_000),   # extraction-keyword mirrors
    sample_up_to(synth_benign,      8_000),
    sample_up_to(shieldlm_benign,  20_000),   # primary filler for B (large pool)
], ignore_index=True)
ben_b = ben_b.drop_duplicates(subset='text')
gap_b = target_b - len(ben_b)
if gap_b > 0:
    ben_b = pd.concat([ben_b, sample_up_to(alpaca_df, gap_b)], ignore_index=True)
ben_b = ben_b.drop_duplicates(subset='text').reset_index(drop=True)
ben_b['label'] = 0
print(f'SLM B: Privilege Escalation Detector benign: {len(ben_b):,} (target {target_b:,})')

SLM B: Privilege Escalation Detector benign: 62,667 (target 62,670)


In [65]:
# SLM A: Instruction & Role Violation Detector benign allocation
# Target: 3 × len(inj_a)
target_a = len(inj_a) * 3
ben_a = pd.concat([
    sample_up_to(neur_benign,       3_000),   # security-domain hard negatives
    sample_up_to(tm_benign,         8_000),   # largest high-quality security benign pool
    sample_up_to(mirror_benign,     4_000),   # CRITICAL: keyword-present benign context
    sample_up_to(synth_benign,      5_000),   # general instruction text
    sample_up_to(shieldlm_benign,  10_000),   # multilingual + tool responses
    sample_up_to(centrepour_benign,   720),   # JailbreakBench goals — hard negatives
], ignore_index=True)
ben_a = ben_a.drop_duplicates(subset='text')
gap_a = target_a - len(ben_a)
if gap_a > 0:
    ben_a = pd.concat([ben_a, sample_up_to(alpaca_df, gap_a)], ignore_index=True)
ben_a = ben_a.drop_duplicates(subset='text').reset_index(drop=True)
ben_a['label'] = 0
print(f'SLM A: Instruction & Role Violation Detector benign: {len(ben_a):,} (target {target_a:,})')

SLM A: Instruction & Role Violation Detector benign: 50,608 (target 50,610)


In [66]:
# SLM C: Obfuscation & Evasion Pattern Detector benign allocation
target_c = len(inj_c) * 3
ben_c = pd.concat([
    sample_up_to(neur_benign,       3_000),
    sample_up_to(tm_benign,         8_000),
    sample_up_to(mirror_benign,     2_000),
    sample_up_to(synth_benign,     15_000),   # large diverse instruction pool
    sample_up_to(shieldlm_benign,  12_000),
    sample_up_to(centrepour_benign,   720),
], ignore_index=True)
ben_c = ben_c.drop_duplicates(subset='text')
gap_c = target_c - len(ben_c)
if gap_c > 0:
    ben_c = pd.concat([ben_c, sample_up_to(alpaca_df, gap_c)], ignore_index=True)
ben_c = ben_c.drop_duplicates(subset='text').reset_index(drop=True)
ben_c['label'] = 0
print(f'SLM C: Obfuscation & Evasion Pattern Detector benign: {len(ben_c):,} (target {target_c:,})')

SLM C: Obfuscation & Evasion Pattern Detector benign: 59,998 (target 60,000)


# Merge, deduplicate, verify 3:1 ratio

In [67]:
def build_dataset(inj_df, ben_df, name):
    """
    Merge injection + benign, cross-deduplicate, enforce 1:1 ratio,
    shuffle, return final DataFrame.
    """
    # Merge
    merged = pd.concat([inj_df, ben_df], ignore_index=True)
    merged = merged.drop_duplicates(subset='text').reset_index(drop=True)

    # Re-check ratio after cross-dedup
    n_inj = (merged['label'] == 1).sum()
    n_ben = (merged['label'] == 0).sum()
    target_ben = n_inj

    if n_ben > target_ben:
        # Trim benign to maintain 1:1
        ben_rows = merged[merged['label'] == 0].sample(n=target_ben, random_state=42)
        inj_rows = merged[merged['label'] == 1]
        merged = pd.concat([inj_rows, ben_rows], ignore_index=True)

    # Final shuffle
    merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)

    n_inj_f = (merged['label'] == 1).sum()
    n_ben_f = (merged['label'] == 0).sum()
    ratio = n_ben_f / n_inj_f
    print(f'{name}: {len(merged):,} rows | injections={n_inj_f:,} | benign={n_ben_f:,} | ratio=1:{ratio:.2f}')
    return merged

raw_a = build_dataset(inj_a, ben_a, 'SLM A: Instruction & Role Violation Detector')
raw_b = build_dataset(inj_b, ben_b, 'SLM B: Privilege Escalation Detector')
raw_c = build_dataset(inj_c, ben_c, 'SLM C: Obfuscation & Evasion Pattern Detector')

SLM A: Instruction & Role Violation Detector: 33,740 rows | injections=16,870 | benign=16,870 | ratio=1:1.00
SLM B: Privilege Escalation Detector: 41,780 rows | injections=20,890 | benign=20,890 | ratio=1:1.00
SLM C: Obfuscation & Evasion Pattern Detector: 40,000 rows | injections=20,000 | benign=20,000 | ratio=1:1.00


# Gemma 3 chat-template formatting

Gemma 3 Instruct uses the `<start_of_turn>` / `<end_of_turn>` chat template.

Each training example is now formatted as:
```text
<bos><start_of_turn>user
{classification_instruction_with_raw_prompt}<end_of_turn>
<start_of_turn>model
INJECTION<end_of_turn>
```
or:
```text
<bos><start_of_turn>user
{classification_instruction_with_raw_prompt}<end_of_turn>
<start_of_turn>model
BENIGN<end_of_turn>
```

Important changes:
- `<bos>` is added explicitly for Gemma 3 compatibility.
- The model answer is appended after `<start_of_turn>model`.
- `label` is retained in every split for evaluation metrics such as precision, recall, F1, and false-positive rate.
- Multiple instruction variants are used to reduce template overfitting.


In [68]:
# Instruction prompt variants
# Using multiple variants helps reduce overfitting to one fixed template.

slm_instructions = {
    'Instruction & Role Violation': [
        (
            "Analyze the following user prompt and determine if it attempts to override "
            "system instructions or hijack the assistant's persona."
            "User Prompt:{prompt}"
            "Respond with exactly one word: INJECTION or BENIGN"
        )
    ],
    'Privilege Escalation': [
        (
            "Analyze the following user prompt and determine if it attempts to extract "
            "system prompts, invoke admin mode, or bypass safety policies."
            "User Prompt:{prompt}"
            "Respond with exactly one word: INJECTION or BENIGN"
        )
    ],
    'Obfuscation & Evasion': [
        (
            "Analyze the following user prompt and determine if it uses encoding tricks, "
            "delimiter injection, or structural evasion."
            "User Prompt:{prompt}"
            "Respond with exactly one word: INJECTION or BENIGN"
        )
    ],
}

ir_instructions = slm_instructions['Instruction & Role Violation']
pe_instructions = slm_instructions['Privilege Escalation']
oe_instructions = slm_instructions['Obfuscation & Evasion']


In [69]:
def format_gemma3(row, system_instruction):
    """
    Produces a fully formatted Gemma 3 training string.
    Includes explicit <bos>, user turn, model turn with gold answer, and <end_of_turn>.
    """
    user_turn = system_instruction.format(prompt=str(row['text']))
    answer = 'INJECTION' if int(row['label']) == 1 else 'BENIGN'
    return (
        f'<bos><start_of_turn>user {user_turn}<end_of_turn>'
        f'<start_of_turn>model{answer}<end_of_turn>'
    )

print('Gemma 3 formatter with explicit <bos> and model answer defined.')

Gemma 3 formatter with explicit <bos> and model answer defined.


In [70]:
def apply_template(df, instructions, slm_name):
    out = df.copy()
    out['formatted_text'] = out.apply(
        lambda r: format_gemma3(r, instructions[-1]),
        axis=1,
    )
    out = out[['formatted_text', 'label']].copy()
    out['label'] = out['label'].astype(int)

    sample = out.iloc[0]['formatted_text']
    assert sample.startswith('<bos><start_of_turn>user'), 'Missing Gemma BOS/user turn.'
    assert '<start_of_turn>model' in sample, 'Missing model answer turn.'
    assert sample.endswith('<end_of_turn>'), 'Missing final end_of_turn.'

    print(f'{slm_name} label distribution:')
    print(out['label'].value_counts().sort_index().rename(index={0: 'BENIGN', 1: 'INJECTION'}))
    print(f'{slm_name} sample formatted row:')
    print(sample[:500], '...')
    return out

fmt_a = apply_template(raw_a, ir_instructions, 'SLM A: Instruction & Role Violation Detector')
fmt_b = apply_template(raw_b, pe_instructions, 'SLM B: Privilege Escalation Detector')
fmt_c = apply_template(raw_c, oe_instructions, 'SLM C: Obfuscation & Evasion Pattern Detector')


SLM A: Instruction & Role Violation Detector label distribution:
label
BENIGN       16870
INJECTION    16870
Name: count, dtype: int64
SLM A: Instruction & Role Violation Detector sample formatted row:
<bos><start_of_turn>user Analyze the following user prompt and determine if it attempts to override system instructions or hijack the assistant's persona.User Prompt:Write a short composing music discussing neural networksRespond with exactly one word: INJECTION or BENIGN<end_of_turn><start_of_turn>modelBENIGN<end_of_turn> ...
SLM B: Privilege Escalation Detector label distribution:
label
BENIGN       20890
INJECTION    20890
Name: count, dtype: int64
SLM B: Privilege Escalation Detector sample formatted row:
<bos><start_of_turn>user Analyze the following user prompt and determine if it attempts to extract system prompts, invoke admin mode, or bypass safety policies.User Prompt:List two points of contrast between the two characters in the story.Respond with exactly one word: INJECTION or

# Stratified 80/10/10 train/val/test split

In [71]:
from sklearn.model_selection import train_test_split

def print_split_stats(split_name, df):
    counts = df['label'].value_counts().sort_index()
    benign = int(counts.get(0, 0))
    injection = int(counts.get(1, 0))
    total = benign + injection
    inj_pct = (injection / total * 100) if total else 0
    print(f'    {split_name}: {total:,} rows | BENIGN={benign:,} | INJECTION={injection:,} | injection={inj_pct:.2f}%')

def stratified_split(df, name):
    """
    Stratified 80/10/10 split.
    Returns (train_df, val_df, test_df), each with formatted_text + label.
    Keeping label enables evaluation: precision, recall, F1, FPR, confusion matrix.
    """
    train_df, temp_df = train_test_split(
        df, test_size=0.20, random_state=42, stratify=df['label']
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
    )

    train_df = train_df[['formatted_text', 'label']].reset_index(drop=True)
    val_df   = val_df  [['formatted_text', 'label']].reset_index(drop=True)
    test_df  = test_df [['formatted_text', 'label']].reset_index(drop=True)

    print(f'{name} split stats:')
    print_split_stats('train', train_df)
    print_split_stats('validation', val_df)
    print_split_stats('test', test_df)
    return train_df, val_df, test_df

train_a, val_a, test_a = stratified_split(fmt_a, 'SLM A: Instruction & Role Violation Detector')
train_b, val_b, test_b = stratified_split(fmt_b, 'SLM B: Privilege Escalation Detector')
train_c, val_c, test_c = stratified_split(fmt_c, 'SLM C: Obfuscation & Evasion Pattern Detector')


SLM A: Instruction & Role Violation Detector split stats:
    train: 26,992 rows | BENIGN=13,496 | INJECTION=13,496 | injection=50.00%
    validation: 3,374 rows | BENIGN=1,687 | INJECTION=1,687 | injection=50.00%
    test: 3,374 rows | BENIGN=1,687 | INJECTION=1,687 | injection=50.00%
SLM B: Privilege Escalation Detector split stats:
    train: 33,424 rows | BENIGN=16,712 | INJECTION=16,712 | injection=50.00%
    validation: 4,178 rows | BENIGN=2,089 | INJECTION=2,089 | injection=50.00%
    test: 4,178 rows | BENIGN=2,089 | INJECTION=2,089 | injection=50.00%
SLM C: Obfuscation & Evasion Pattern Detector split stats:
    train: 32,000 rows | BENIGN=16,000 | INJECTION=16,000 | injection=50.00%
    validation: 4,000 rows | BENIGN=2,000 | INJECTION=2,000 | injection=50.00%
    test: 4,000 rows | BENIGN=2,000 | INJECTION=2,000 | injection=50.00%


# Convert to HuggingFace DatasetDict and push to Hub

Each dataset is pushed as a private repo under `hirushafernando/fyp-slm-a`, etc.  
Each repo has three splits: `train`, `validation`, `test`.

**Columns in each split:**
- `formatted_text` — Gemma 3 formatted training text for `SFTTrainer`
- `label` — integer label retained for evaluation (`0 = BENIGN`, `1 = INJECTION`)

In `SFTTrainer`, use `dataset_text_field='formatted_text'`.


In [72]:
def make_dataset_dict(train_df, val_df, test_df):
    return DatasetDict({
        'train':      Dataset.from_pandas(train_df, preserve_index=False),
        'validation': Dataset.from_pandas(val_df,   preserve_index=False),
        'test':       Dataset.from_pandas(test_df,  preserve_index=False),
    })

ds_a = make_dataset_dict(train_a, val_a, test_a)
ds_b = make_dataset_dict(train_b, val_b, test_b)
ds_c = make_dataset_dict(train_c, val_c, test_c)

print('DatasetDicts created:')
print('  SLM A: Instruction & Role Violation Detector - ', ds_a)
print('  SLM B: Privilege Escalation Detector - ', ds_b)
print('  SLM C: Obfuscation & Evasion Pattern Detector - ', ds_c)

DatasetDicts created:
  SLM A: Instruction & Role Violation Detector -  DatasetDict({
    train: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 26992
    })
    validation: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 3374
    })
    test: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 3374
    })
})
  SLM B: Privilege Escalation Detector -  DatasetDict({
    train: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 33424
    })
    validation: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 4178
    })
    test: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 4178
    })
})
  SLM C: Obfuscation & Evasion Pattern Detector -  DatasetDict({
    train: Dataset({
        features: ['formatted_text', 'label'],
        num_rows: 32000
    })
    validation: Dataset({
        features: ['formatted_text', 'label'],
        num_rows:

In [74]:
# Push to HuggingFace Hub
# Repos created: hirushafernando/fyp-slm-a, fyp-slm-b, fyp-slm-c
# All private.

HF_USERNAME = 'hirushafernando'   # change if your HF username differs

REPO_A = f'{HF_USERNAME}/slm-shield-ds-role-and-instruction-violation'
REPO_B = f'{HF_USERNAME}/slm-shield-ds-privilege-escalation'
REPO_C = f'{HF_USERNAME}/slm-shield-ds-obfuscation-and-evasion-patterns'

print(f'Pushing SLM A: Instruction & Role Violation Detector dataset to {REPO_A} ...')
ds_a.push_to_hub(REPO_A, private=True, token=HF_TOKEN)
print('  Done.')

print(f'Pushing SLM B: Privilege Escalation Detector dataset to {REPO_B} ...')
ds_b.push_to_hub(REPO_B, private=True, token=HF_TOKEN)
print('  Done.')

print(f'Pushing SLM C: Obfuscation & Evasion Pattern Detector dataset to {REPO_C} ...')
ds_c.push_to_hub(REPO_C, private=True, token=HF_TOKEN)
print('  Done.')

print('\n✅ All three datasets pushed to HuggingFace Hub.')

Pushing SLM A: Instruction & Role Violation Detector dataset to hirushafernando/slm-shield-ds-role-and-instruction-violation ...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 14.94ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   9%|▉         |  529kB / 5.98MB, 49.4kB/s  
Processing Files (0 / 1):  18%|█▊        | 1.06MB / 5.98MB, 95.1kB/s  
Processing Files (0 / 1):  27%|██▋       | 1.59MB / 5.98MB,  138kB/s  
Processing Files (0 / 1):  35%|███▌      | 2.12MB / 5.98MB,  181kB/s  
Processing Files (0 / 1):  44%|████▍     | 2.65MB / 5.98MB,  226kB/s  
Processing Files (0 / 1):  62%|██████▏   | 3.71MB / 5.98MB,  319kB/s  
Processing Files (0 / 1):  71%|███████   | 4.23MB / 5.98MB,  360kB/s  
Processing Files (0 / 1):  80%|███████▉  | 4.76MB / 5.98MB,  397kB/s  
Processing Files (0 / 1):  89%|████████▊ | 5.29MB / 5.98MB,  432kB/s  
Processing Files (0 / 1):  97%|█████████▋| 5.82MB / 5.98MB,  466kB/s  
Processing Files (1 / 1): 100

  Done.
Pushing SLM B: Privilege Escalation Detector dataset to hirushafernando/slm-shield-ds-privilege-escalation ...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 17.51ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   8%|▊         |  536kB / 6.47MB, 50.3kB/s  
Processing Files (0 / 1):  17%|█▋        | 1.07MB / 6.47MB, 96.8kB/s  
Processing Files (0 / 1):  25%|██▍       | 1.61MB / 6.47MB,  142kB/s  
Processing Files (0 / 1):  33%|███▎      | 2.14MB / 6.47MB,  185kB/s  
Processing Files (0 / 1):  41%|████▏     | 2.68MB / 6.47MB,  230kB/s  
Processing Files (0 / 1):  66%|██████▋   | 4.29MB / 6.47MB,  365kB/s  
Processing Files (0 / 1):  75%|███████▍  | 4.82MB / 6.47MB,  402kB/s  
Processing Files (0 / 1):  83%|████████▎ | 5.36MB / 6.47MB,  434kB/s  
Processing Files (0 / 1):  91%|█████████ | 5.89MB / 6.47MB,  468kB/s  
Processing Files (0 / 1):  99%|█████████▉| 6.43MB / 6.47MB,  502kB/s  
Processing Files (1 / 1): 100

  Done.
Pushing SLM C: Obfuscation & Evasion Pattern Detector dataset to hirushafernando/slm-shield-ds-obfuscation-and-evasion-patterns ...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 16.51ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   8%|▊         |  528kB / 6.45MB, 48.7kB/s  
Processing Files (0 / 1):  16%|█▋        | 1.06MB / 6.45MB, 91.1kB/s  
Processing Files (0 / 1):  33%|███▎      | 2.11MB / 6.45MB,  182kB/s  
Processing Files (0 / 1):  57%|█████▋    | 3.70MB / 6.45MB,  315kB/s  
Processing Files (0 / 1):  66%|██████▌   | 4.23MB / 6.45MB,  352kB/s  
Processing Files (0 / 1):  74%|███████▎  | 4.76MB / 6.45MB,  388kB/s  
Processing Files (0 / 1):  98%|█████████▊| 6.34MB / 6.45MB,  515kB/s  
Processing Files (1 / 1): 100%|██████████| 6.45MB / 6.45MB,  418kB/s  
Processing Files (1 / 1): 100%|██████████| 6.45MB / 6.45MB,  364kB/s  
New Data Upload: 100%|██████████| 6.45MB / 6.45MB,  364kB/s  
Uploading the dataset shards: 100%|███

  Done.

✅ All three datasets pushed to HuggingFace Hub.


In [76]:
# Verification — reload from Hub and spot-check
check_a = load_dataset(REPO_A, token=HF_TOKEN)
check_b = load_dataset(REPO_B, token=HF_TOKEN)
check_c = load_dataset(REPO_C, token=HF_TOKEN)

datasets_data = [
    ('SLM A: Instruction & Role Violation Detector', check_a), 
    ('SLM B: Privilege Escalation Detector', check_b), 
    ('SLM C: Obfuscation & Evasion Pattern Detector', check_c)
]

for name, ds in datasets_data:
    print(f'{name}:')
    print('  Columns:', ds.column_names)
    for split in ['train', 'validation', 'test']:
        labels = ds[split].to_pandas()['label'].value_counts().sort_index().to_dict()
        print(
            f'  {split}: {len(ds[split]):,} rows | cols: {ds[split].column_names} | '
            f'BENIGN={labels.get(0, 0):,} | INJECTION={labels.get(1, 0):,}'
        )
    print('  Sample train[0]:')
    print('  ', ds['train'][0]['formatted_text'][:500], '...')


D:\Python\SLM-Shield\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hesha\.cache\huggingface\hub\datasets--hirushafernando--slm-shield-ds-role-and-instruction-violation. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 3374/3374 [00:00<00:00, 36409.90 exampl

SLM A: Instruction & Role Violation Detector:
  Columns: {'train': ['formatted_text', 'label'], 'validation': ['formatted_text', 'label'], 'test': ['formatted_text', 'label']}
  train: 26,992 rows | cols: ['formatted_text', 'label'] | BENIGN=13,496 | INJECTION=13,496
  validation: 3,374 rows | cols: ['formatted_text', 'label'] | BENIGN=1,687 | INJECTION=1,687
  test: 3,374 rows | cols: ['formatted_text', 'label'] | BENIGN=1,687 | INJECTION=1,687
  Sample train[0]:
   <bos><start_of_turn>user Analyze the following user prompt and determine if it attempts to override system instructions or hijack the assistant's persona.User Prompt:Q1: Which defensive move in baseball involves a fielder touching a ball while it is still in the air after it has been batted by a batter?
A1: An outfield catch (or "catching a fly ball").

Q2: When climbing a rock face using natural features without ropes for protection (such as cracks or ledges), what is this sport called?
A2: Fre ...
SLM B: Privilege Escala

---
## Summary

| Dataset | Columns | Train | Val | Test | Total |
|---------|---------|-------|-----|------|-------|
| `fyp-slm-a` | `formatted_text`, `label` | 80% | 10% | 10% | ~67k |
| `fyp-slm-b` | `formatted_text`, `label` | 80% | 10% | 10% | ~84k |
| `fyp-slm-c` | `formatted_text`, `label` | 80% | 10% | 10% | ~80k |

**Next notebook → `02_finetune_slm_a.ipynb`**  
Load `hirushafernando/fyp-slm-a`, load `google/gemma-3-1b-it` or `unsloth/gemma-3-1b-it-unsloth-bnb-4bit` with 4-bit QLoRA, then run `SFTTrainer`.

In `SFTTrainer` pass:
```python
dataset_text_field='formatted_text'
```

No `formatting_func` needed because the Gemma 3 chat template is already included.

During evaluation, use the retained `label` column to compute:
- precision
- recall
- F1-score
- false-positive rate
- confusion matrix
